# Feature — Content-Profile Similarity (enhanced Feature 3)

**Purpose.** Personalise through the *text* of the products a user has touched, not only their IDs. Steps 2–4b personalise through exact item / brand / category IDs (`uc_*`, `ubrand_*`, `ucat*_*`) and through a 64-d LSA profile (`u_sem_lsa_7d`, `u_sem_lsa_all`). Exact IDs miss two phone cases from different shops and leaf categories that share most of their title terms, and the LSA profile keeps only a small part of the TF-IDF variance (Step 4d lists the scalar semantic profiles as drop candidates). This notebook scores every candidate against the user's history in the sparse TF-IDF space itself.

**What it adds** (columns in §3), compared with one cosine to one averaged profile:
1. richer item text: name terms plus field-prefixed brand-name and level-3/4 category-name terms;
2. nearest-neighbour similarity (`cps_max`), so a user with several interests is not blurred into one average;
3. a query-conditioned profile (`cps_query`) that re-weights the history by its relevance to the current query;
4. funnel-stage profiles (cart + order, order only) and the age of the best-matching history product, aimed at the purchase cut-point;
5. long (90-day) and short (1-day) horizons as separate columns;
6. `cps_top3_new`, similarity to the nearest history products *other than the candidate itself*, which repeat encounters cannot inflate.

**Contract.** Run after Steps 1–4b. This notebook does **not** modify `features.parquet` or any earlier output. It writes `processed/feature_content_profile/`: the feature table keyed by `user, wid` (in `candidates.parquet` row order) and, per run, the metrics and a manifest. Model comparisons use the train and validation splits only; earlier steps already used the test split for feature analysis, so it is not used here to select features.

Set `JD_SMOKE=1` for a quick local run on ~2% of users. The full run needs a High-RAM Colab runtime (TF-IDF over all slate and history products); no GPU is needed.

In [ ]:
# In Colab, run this once if dependencies are missing:
# %pip install lightgbm pyarrow pandas numpy scipy
import os, json, time, gc, platform, importlib.metadata
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import scipy.sparse as sp
import pyarrow as pa, pyarrow.compute as pc, pyarrow.csv as pacsv

try:                                    # Colab: data lives on the shared Drive
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/JDSearch/Raw Data')
except ImportError:                     # local run: set JD_DATA_DIR or use the default download location
    DATA_DIR = Path(os.environ.get('JD_DATA_DIR', '~/Downloads/JDsearch')).expanduser()
PROCESSED = DATA_DIR / 'processed'

SMOKE = os.environ.get('JD_SMOKE') == '1'      # local smoke test: ~2% of users
SEED = 42
SEEDS = [42, 43, 44]                           # LightGBM seeds for the matched comparisons
TREES = 300                                    # equal tree budget for every variant, as in Step 8
BOOTSTRAPS = 2000
OUT = PROCESSED / 'feature_content_profile'
RUN = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + ('_smoke' if SMOKE else '')
RUN_DIR = OUT / 'runs' / RUN
RUN_DIR.mkdir(parents=True, exist_ok=False)
FEATURE_FILE = OUT / f"content_profile_features{'_smoke' if SMOKE else ''}.parquet"
print('data:', DATA_DIR, '| run:', RUN)

## 1. Candidates, history and the test query

Each user has exactly one slate (the test query), so `user` keys both the history and the slate. Candidates keep the `candidates.parquet` row order, which is also the row order of `features.parquet`. History events are kept in chronological order (`pos`) and include follows (`type == 3`), which count as a click-strength signal below.

In [ ]:
users = pd.read_parquet(PROCESSED / 'users.parquet', columns=['user', 'query'])
nU = len(users)
assert (users['user'].to_numpy() == np.arange(nU)).all()
in_scope = np.random.default_rng(SEED + 2).random(nU) < 0.02 if SMOKE else np.ones(nU, bool)

cands = pd.read_parquet(PROCESSED / 'candidates.parquet', columns=['user', 'wid', 'label'])
cands = cands[in_scope[cands['user'].to_numpy()]].reset_index(drop=True)
hist = pd.read_parquet(PROCESSED / 'events.parquet', columns=['user', 'pos', 'wid', 'type', 'age_days'])
hist = hist[in_scope[hist['user'].to_numpy()]].sort_values(['user', 'pos'], kind='stable').reset_index(drop=True)
assert (np.diff(cands['user'].to_numpy()) >= 0).all(), 'candidates.parquet is expected in user order'
assert hist['type'].isin([0, 1, 2, 3]).all() and hist['age_days'].ge(0).all()
print(f"users {in_scope.sum():,} | candidate rows {len(cands):,} | history events {len(hist):,}")